In [1]:
import os

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
# Loading Training Data and testing data

folder_path_Train = 'train'
folder_path_Test = 'test'

def load_data(folder):
    reviews = []
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        with open(file_path, "r", encoding="utf-8") as f:     #'with open' is used to make sure that the object 'f' is closed after usage
                                                              # 'r' is used for the reading mode. 'utf-8'encoding ensures the special characters are managed correctly
            reviews.append(f.read())
    return reviews

positive_reviews_Train = load_data(os.path.join(folder_path_Train,"pos"))
negative_reviews_Train = load_data(os.path.join(folder_path_Train,"neg"))

positive_reviews_Test = load_data(os.path.join(folder_path_Test,"pos"))
negative_reviews_Test = load_data(os.path.join(folder_path_Test,"neg"))


In [4]:
all_reviews_training = positive_reviews_Train+negative_reviews_Train     #The positive and negative reviews are combined for easy access in the preproccing part
all_reviews_testing = positive_reviews_Test+negative_reviews_Test

In [5]:
train_labels = [1] * len(positive_reviews_Train) + [0]* len(negative_reviews_Train)
test_labels = [1]* len(positive_reviews_Test) + [0] * len(negative_reviews_Test)

#These labels are used as y variable in the training and the testing phase

In [6]:
import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dulin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dulin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
def preprocess(text):
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z\s]", "", text)  
    words = text.split()
    words = [word for word in words if word not in stopwords.words("english")]  
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words] 
    return " ".join(words)

In [8]:
train_reviews_preprocessd = [preprocess(review) for review in all_reviews_training] 
test_reviews_preprocessd = [preprocess(review) for review in all_reviews_testing] 

C:\Users\dulin\AppData\Local\Temp\ipykernel_140768\1681393399.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(train_reviews_preprocessd)
X_test = vectorizer.transform(test_reviews_preprocessd)

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, train_labels)

c:\Users\dulin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [11]:
y_pred = model.predict(X_test)

In [13]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(test_labels, y_pred)
accuracy

0.8596

In [ ]:
import joblib

joblib.dump(vectorizer, "vectorizer.pkl")  
joblib.dump(model, "sentiment_model.pkl")

['sentiment_model.pkl']

In [38]:
vectorizer = joblib.load("vectorizer.pkl")  
model = joblib.load("sentiment_model.pkl")

In [39]:
def new_prediction(review):
    preproccessed_text_new = preprocess(review)
    vectorized_review = vectorizer.transform([preproccessed_text_new])
    prediction = model.predict(vectorized_review)  # Predict sentiment
    return "Positive" if prediction[0] == 1 else "Negative"



In [40]:
new_review = "Beautiful Movie. Will recommend it to my friends"


print(new_prediction(new_review))

ValueError: X has 4 features, but LogisticRegression is expecting 129016 features as input.